In [1]:
import pandas as pd
import sys
sys.path.append("src")
from loan_default_ml_pipeline.ingestion.load_data import load_raw_data

df = load_raw_data()

In [2]:
import importlib
import loan_default_ml_pipeline.features.build_features as bf
importlib.reload(bf)

<module 'loan_default_ml_pipeline.features.build_features' from '/Users/matthewventura/Projects/loan-default-ml-pipeline/src/loan_default_ml_pipeline/features/build_features.py'>

In [3]:
from pathlib import Path
print(Path("../src/loan_default_ml_pipeline/features/build_features.py").read_text())

import pandas as pd

# -- Constants ----------------------------------
LEAKAGE_COLS = [
    "paid_total",
    "paid_principal",
    "paid_interest",
    "paid_late_fees",
    "balance"
]

DEFAULT_STATUSES = [
    "Charged Off",
    "Late (31-120 days)",
    "Late (16-30 days)"
]

KEEP_STATUSES = [
    "Charged Off",
    "Late (31-120 days)",
    "Late (16-30 days)",
    "Fully Paid"
]

JOINT_COLUMNS_INT = [
    "annual_income_joint",
    "debt_to_income_joint",
    "verification_income_joint"
]

MONTHS_SINCE_COLS = [
    "months_since_last_delinq",
    "months_since_90d_late",
    "months_since_last_credit_inquiry"
]

# -- Step 1: Drop leakage columns -----------------
def drop_leakage_cols(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=LEAKAGE_COLS, errors="ignore")

# -- Step 2: Encode target label ------------------
def encode_target(df: pd.DataFrame) -> pd.DataFrame:
    df["default"] = df["loan_status"].isin(DEFAULT_STATUSES).astype(int)
    # binary bool testing if

In [4]:
from loan_default_ml_pipeline.features.build_features import drop_leakage_cols, LEAKAGE_COLS

df_clean = drop_leakage_cols(df)

print("Columns removed:", [c for c in LEAKAGE_COLS if c not in df_clean.columns])
print("Shape raw:", df.shape)
print("Shape trimmed:", df_clean.shape)

Columns removed: ['paid_total', 'paid_principal', 'paid_interest', 'paid_late_fees', 'balance']
Shape raw: (10000, 55)
Shape trimmed: (10000, 50)


In [5]:
from loan_default_ml_pipeline.features.build_features import encode_target

df_clean = encode_target(df_clean)

print("Default value counts:")
print(df_clean["default"].value_counts())
print("\nSample - loan_status vs default flag:")
print(df_clean[["loan_status", "default"]].drop_duplicates().sort_values("default"))

Default value counts:
default
0    9889
1     111
Name: count, dtype: int64

Sample - loan_status vs default flag:
            loan_status  default
0               Current        0
18           Fully Paid        0
37      In Grace Period        0
224  Late (31-120 days)        1
387         Charged Off        1
484   Late (16-30 days)        1


In [6]:
importlib.reload(bf)
from loan_default_ml_pipeline.features.build_features import filter_ambiguous

df_clean = filter_ambiguous(df_clean)

print("Shape after filter:", df_clean.shape)
print("\nDefault value counts:")
print(df_clean["default"].value_counts())
print("\nRemaining columns include loan_status:", "loan_status" in df_clean.columns)

Shape after filter: (558, 50)

Default value counts:
default
0    447
1    111
Name: count, dtype: int64

Remaining columns include loan_status: False


In [7]:
importlib.reload(bf)
from loan_default_ml_pipeline.features.build_features import handle_joint_nulls, JOINT_COLUMNS_INT

df_clean = handle_joint_nulls(df_clean)

print("is_joint_app value counts:")
print(df_clean["is_joint_app"].value_counts())
print("\nNulls remaining in joint columns:")
print(df_clean[JOINT_COLUMNS_INT].isnull().sum())

is_joint_app value counts:
is_joint_app
0    479
1     79
Name: count, dtype: int64

Nulls remaining in joint columns:
annual_income_joint          0
debt_to_income_joint         0
verification_income_joint    0
dtype: int64


In [8]:
importlib.reload(bf)
from loan_default_ml_pipeline.features.build_features import handle_delinq_nulls, MONTHS_SINCE_COLS

df_clean = handle_delinq_nulls(df_clean)

print("ever_delinquent value counts:")
print(df_clean["ever_delinquent"].value_counts())
print("\never_90d_late value counts:")
print(df_clean["ever_90d_late"].value_counts())
print(df_clean[MONTHS_SINCE_COLS].isnull().sum())

ever_delinquent value counts:
ever_delinquent
0    310
1    248
Name: count, dtype: int64

ever_90d_late value counts:
ever_90d_late
0    437
1    121
Name: count, dtype: int64
months_since_last_delinq            0
months_since_90d_late               0
months_since_last_credit_inquiry    0
dtype: int64


In [9]:
importlib.reload(bf)
from loan_default_ml_pipeline.features.build_features import encode_categoricals

df_clean = encode_categoricals(df_clean)

print("Final shape:", df_clean.shape)
print("\nAny nulls remaining:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print("\nAll dtypes object (should be none):")
print(df_clean.select_dtypes(include="object").columns.tolist())


Final shape: (558, 62)

Any nulls remaining:
Series([], dtype: int64)

All dtypes object (should be none):
[]


In [10]:
# Main test
importlib.reload(bf)
from loan_default_ml_pipeline.features.build_features import build_features

df_featured = build_features(df)

print("Input shape:", df.shape)
print("Output shape:", df_featured.shape)
print("Null remaining:", df_featured.isnull().sum().sum())
print("Object columns:", df_featured.select_dtypes(include="object").columns.tolist())
print("Target distribution:")
print(df_featured["default"].value_counts())

Input shape: (10000, 55)
Output shape: (558, 62)
Null remaining: 0
Object columns: []
Target distribution:
default
0    447
1    111
Name: count, dtype: int64


In [11]:
from pathlib import Path

output_path = Path("../data/processed/loans_featured.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df_featured.to_csv(output_path, index=False)

print(f"Saved to {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Saved to ../data/processed/loans_featured.csv
File size: 145.7 KB
